In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
import warnings

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
with open('dataset_stats.json', 'r') as f:
    stats = json.load(f)

num_classes = stats['num_classes']
img_size = stats['img_size']
idx_to_breed = {int(k): v for k, v in stats['idx_to_breed'].items()}
class_weight_dict = {int(k): v for k, v in stats['class_weight_dict'].items()}

print(f"✓ Number of classes: {num_classes}")
print(f"✓ Image size: {img_size}x{img_size}")

train_df = pd.read_csv('train_split.csv')
val_df = pd.read_csv('val_split.csv')
test_df = pd.read_csv('test_split.csv')

print(f"✓ Train: {len(train_df)} images")
print(f"✓ Val: {len(val_df)} images")
print(f"✓ Test: {len(test_df)} images")

In [ ]:
BASE_PATH = '../dog-breed-identification/dog-breed-identification/'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')

train_df['filepath'] = train_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))
val_df['filepath'] = val_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))
test_df['filepath'] = test_df['id'].apply(lambda x: os.path.join(TRAIN_PATH, f"{x}.jpg"))

train_df['label_str'] = train_df['breed'].astype(str)
val_df['label_str'] = val_df['breed'].astype(str)
test_df['label_str'] = test_df['breed'].astype(str)

batch_size = 32

# ImageNet preprocessing (built into Keras applications)
train_datagen = ImageDataGenerator(
    preprocessing_function=keras.applications.resnet50.preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=keras.applications.resnet50.preprocess_input
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label_str',
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print(f"✓ Using ImageNet preprocessing")
print(f"✓ Batch size: {batch_size}")
print(f"✓ Train batches: {len(train_generator)}")

In [ ]:
def train_transfer_model(model_name, base_model, num_epochs=25):
    """Complete training pipeline for transfer learning"""
    
    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"{'='*70}")
    
    # Build model
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)  # Freeze base model
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    # Count parameters
    total_params = model.count_params()
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
    frozen_params = total_params - trainable_params
    
    print(f"✓ Model: {model_name}")
    print(f"✓ Total parameters: {total_params:,}")
    print(f"✓ Trainable parameters: {trainable_params:,}")
    print(f"✓ Frozen parameters: {frozen_params:,}")
    print(f"✓ Number of classes: {num_classes}")
    
    # Compile
    learning_rate = 0.001
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"✓ Learning rate: {learning_rate}")
    print(f"✓ Epochs: {num_epochs}")
    
    # Callbacks
    callbacks = [
        ModelCheckpoint(
            f'{model_name}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    # Train
    start_time = time.time()
    
    history = model.fit(
        train_generator,
        epochs=num_epochs,
        validation_data=val_generator,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    
    training_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print(f"{model_name} Training Complete!")
    print(f"{'='*70}")
    print(f"Training Time: {training_time/60:.2f} minutes")
    
    # Save history
    history_df = pd.DataFrame({
        'epoch': range(1, len(history.history['loss']) + 1),
        'train_loss': history.history['loss'],
        'train_acc': [acc * 100 for acc in history.history['accuracy']],
        'val_loss': history.history['val_loss'],
        'val_acc': [acc * 100 for acc in history.history['val_accuracy']],
        'lr': history.history['lr']
    })
    
    history_df.to_csv(f'{model_name}_history.csv', index=False)
    
    best_epoch = history_df['val_acc'].idxmax() + 1
    best_val_acc = history_df['val_acc'].max()
    
    print(f"Best Val Acc: {best_val_acc:.2f}% (Epoch {best_epoch})")
    
    # Evaluate on test set
    model = keras.models.load_model(f'{model_name}_best.h5')
    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    test_acc *= 100
    
    print(f"Test Acc: {test_acc:.2f}%")
    
    # Calculate Top-5 Accuracy
    print("\nCalculating Top-5 Accuracy...")
    test_generator.reset()
    y_true = test_generator.classes
    y_pred_probs = model.predict(test_generator, verbose=1)
    
    top5_correct = 0
    for i in range(len(y_true)):
        top5_preds = np.argsort(y_pred_probs[i])[-5:]
        if y_true[i] in top5_preds:
            top5_correct += 1
    
    top5_accuracy = (top5_correct / len(y_true)) * 100
    print(f"Top-5 Accuracy: {top5_accuracy:.2f}%")
    
    # Save summary
    summary = {
        'model': model_name,
        'total_params': int(total_params),
        'trainable_params': int(trainable_params),
        'frozen_params': int(frozen_params),
        'epochs_trained': len(history_df),
        'best_epoch': int(best_epoch),
        'best_val_acc': float(best_val_acc),
        'test_acc': float(test_acc),
        'top5_acc': float(top5_accuracy),
        'training_time_minutes': float(training_time / 60)
    }
    
    with open(f'{model_name}_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    return history_df, summary, model

print("✓ Training function defined")

In [ ]:
base_resnet50 = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(img_size, img_size, 3)
)

# Freeze all layers
base_resnet50.trainable = False

print("✓ Loaded ResNet50 pretrained on ImageNet")
print("✓ Frozen all pretrained layers")

# Train ResNet50
resnet50_history, resnet50_summary, resnet50_model = train_transfer_model(
    'resnet50_transfer', 
    base_resnet50, 
    num_epochs=25
)

In [ ]:
# Load pretrained EfficientNetB0
base_efficientnet = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(img_size, img_size, 3)
)

# Freeze all layers
base_efficientnet.trainable = False

print("✓ Loaded EfficientNetB0 pretrained on ImageNet")
print("✓ Frozen all pretrained layers")

# Train EfficientNetB0
efficientnet_history, efficientnet_summary, efficientnet_model = train_transfer_model(
    'efficientnet_transfer', 
    base_efficientnet, 
    num_epochs=25
)

In [ ]:
# ResNet50 curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(resnet50_history['epoch'], resnet50_history['train_loss'], 
             label='Train Loss', linewidth=2)
axes[0].plot(resnet50_history['epoch'], resnet50_history['val_loss'], 
             label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('ResNet50 Transfer - Loss Curves', fontweight='bold', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(resnet50_history['epoch'], resnet50_history['train_acc'], 
             label='Train Acc', linewidth=2)
axes[1].plot(resnet50_history['epoch'], resnet50_history['val_acc'], 
             label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontweight='bold')
axes[1].set_title('ResNet50 Transfer - Accuracy Curves', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('resnet50_transfer_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: resnet50_transfer_curves.png")

# EfficientNetB0 curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(efficientnet_history['epoch'], efficientnet_history['train_loss'], 
             label='Train Loss', linewidth=2)
axes[0].plot(efficientnet_history['epoch'], efficientnet_history['val_loss'], 
             label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('EfficientNetB0 Transfer - Loss Curves', fontweight='bold', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(efficientnet_history['epoch'], efficientnet_history['train_acc'], 
             label='Train Acc', linewidth=2)
axes[1].plot(efficientnet_history['epoch'], efficientnet_history['val_acc'], 
             label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontweight='bold')
axes[1].set_title('EfficientNetB0 Transfer - Accuracy Curves', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('efficientnet_transfer_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: efficientnet_transfer_curves.png")

# ============================================================================
# 7. COMPARE BOTH TRANSFER LEARNING MODELS
# ============================================================================
print("\n[7] Comparing Transfer Learning Models...")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Validation Loss comparison
axes[0].plot(resnet50_history['epoch'], resnet50_history['val_loss'], 
             label='ResNet50', linewidth=2, alpha=0.8)
axes[0].plot(efficientnet_history['epoch'], efficientnet_history['val_loss'], 
             label='EfficientNetB0', linewidth=2, alpha=0.8)
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Validation Loss', fontweight='bold')
axes[0].set_title('Transfer Learning - Validation Loss Comparison', fontweight='bold', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Validation Accuracy comparison
axes[1].plot(resnet50_history['epoch'], resnet50_history['val_acc'], 
             label='ResNet50', linewidth=2, alpha=0.8)
axes[1].plot(efficientnet_history['epoch'], efficientnet_history['val_acc'], 
             label='EfficientNetB0', linewidth=2, alpha=0.8)
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Validation Accuracy (%)', fontweight='bold')
axes[1].set_title('Transfer Learning - Validation Accuracy Comparison', fontweight='bold', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('transfer_learning_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: transfer_learning_comparison.png")

# Print comparison
print("\nTransfer Learning Models Comparison:")
print(f"{'Metric':<25} {'ResNet50':>15} {'EfficientNetB0':>18}")
print("-" * 65)
print(f"{'Best Val Accuracy':<25} {resnet50_summary['best_val_acc']:>14.2f}% {efficientnet_summary['best_val_acc']:>17.2f}%")
print(f"{'Test Accuracy':<25} {resnet50_summary['test_acc']:>14.2f}% {efficientnet_summary['test_acc']:>17.2f}%")
print(f"{'Top-5 Accuracy':<25} {resnet50_summary['top5_acc']:>14.2f}% {efficientnet_summary['top5_acc']:>17.2f}%")
print(f"{'Trainable Params':<25} {resnet50_summary['trainable_params']:>15,} {efficientnet_summary['trainable_params']:>18,}")
print(f"{'Training Time (min)':<25} {resnet50_summary['training_time_minutes']:>14.2f}m {efficientnet_summary['training_time_minutes']:>17.2f}m")

In [ ]:
try:
    # Load all model summaries
    with open('baseline_cnn_summary.json', 'r') as f:
        baseline_summary = json.load(f)
    with open('resnet_scratch_summary.json', 'r') as f:
        resnet_scratch_summary = json.load(f)
    
    # Create comparison table
    all_models = {
        'Baseline CNN': baseline_summary,
        'ResNet-18 (Scratch)': resnet_scratch_summary,
        'ResNet50 (Transfer)': resnet50_summary,
        'EfficientNetB0 (Transfer)': efficientnet_summary
    }
    
    print("\n" + "=" * 100)
    print("COMPREHENSIVE MODEL COMPARISON")
    print("=" * 100)
    print(f"{'Model':<30} {'Test Acc':>12} {'Top-5 Acc':>12} {'Params':>15} {'Time (min)':>12}")
    print("-" * 100)
    
    for model_name, summary in all_models.items():
        test_acc = summary.get('test_acc', 0)
        top5_acc = summary.get('top5_acc', 0)
        params = summary.get('total_params', 0)
        time_min = summary.get('training_time_minutes', 0)
        print(f"{model_name:<30} {test_acc:>11.2f}% {top5_acc:>11.2f}% {params:>15,} {time_min:>11.2f}m")
    
    print("=" * 100)
    
    # Visualize overall comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    models_list = list(all_models.keys())
    test_accs = [all_models[m]['test_acc'] for m in models_list]
    top5_accs = [all_models[m].get('top5_acc', 0) for m in models_list]
    params = [all_models[m]['total_params']/1e6 for m in models_list]
    times = [all_models[m]['training_time_minutes'] for m in models_list]
    
    # Test Accuracy
    axes[0, 0].barh(models_list, test_accs, color='steelblue', alpha=0.8)
    axes[0, 0].set_xlabel('Test Accuracy (%)', fontweight='bold')
    axes[0, 0].set_title('Test Accuracy Comparison', fontweight='bold', fontsize=14)
    axes[0, 0].grid(axis='x', alpha=0.3)
    for i, v in enumerate(test_accs):
        axes[0, 0].text(v + 1, i, f'{v:.2f}%', va='center', fontweight='bold')
    
    # Top-5 Accuracy
    axes[0, 1].barh(models_list, top5_accs, color='coral', alpha=0.8)
    axes[0, 1].set_xlabel('Top-5 Accuracy (%)', fontweight='bold')
    axes[0, 1].set_title('Top-5 Accuracy Comparison', fontweight='bold', fontsize=14)
    axes[0, 1].grid(axis='x', alpha=0.3)
    for i, v in enumerate(top5_accs):
        axes[0, 1].text(v + 1, i, f'{v:.2f}%', va='center', fontweight='bold')
    
    # Parameters
    axes[1, 0].barh(models_list, params, color='lightgreen', alpha=0.8)
    axes[1, 0].set_xlabel('Parameters (Millions)', fontweight='bold')
    axes[1, 0].set_title('Model Size Comparison', fontweight='bold', fontsize=14)
    axes[1, 0].grid(axis='x', alpha=0.3)
    for i, v in enumerate(params):
        axes[1, 0].text(v + 1, i, f'{v:.2f}M', va='center', fontweight='bold')
    
    # Training Time
    axes[1, 1].barh(models_list, times, color='plum', alpha=0.8)
    axes[1, 1].set_xlabel('Training Time (minutes)', fontweight='bold')
    axes[1, 1].set_title('Training Time Comparison', fontweight='bold', fontsize=14)
    axes[1, 1].grid(axis='x', alpha=0.3)
    for i, v in enumerate(times):
        axes[1, 1].text(v + 2, i, f'{v:.1f}m', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('all_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved: all_models_comparison.png")
    
except FileNotFoundError as e:
    print(f"⚠ Some model summaries not found: {e}")
    print("Run all previous notebooks (03 and 04) for complete comparison.")